# Validation for new pipline

# 0.Import

In [24]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import loadmat

SUBJECTS = ["S00775"]

DATA_ROOT = Path(
    "/mnt/newStor/paros/paros_WORK/xintong/data/column_sample_data"
)

TEST_ROOT = Path(
    "/mnt/newStor/paros/paros_WORK/xintong/test"
)

VALIDATION_ROOT = Path(
    "/mnt/newStor/paros/paros_WORK/xintong/code/"
    "cortical_trajectory_analysis/validation"
)

RESULTS_ROOT = VALIDATION_ROOT / "results"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print("Subjects:", SUBJECTS)
print("Validation results:", RESULTS_ROOT)

Subjects: ['S00775']
Validation results: /mnt/newStor/paros/paros_WORK/xintong/code/cortical_trajectory_analysis/validation/results


## a.Create out put path and Validation Summary

In [25]:
subject_paths = {}

for subject in SUBJECTS:
    subject_paths[subject] = {
        "data": DATA_ROOT / subject,
        "new": TEST_ROOT / "new_pipeline" / subject,
        "comparison": TEST_ROOT / "comparison" / subject,
        "results": RESULTS_ROOT / subject,
    }

    subject_paths[subject]["results"].mkdir(
        parents=True,
        exist_ok=True,
    )

subject_paths

{'S00775': {'data': PosixPath('/mnt/newStor/paros/paros_WORK/xintong/data/column_sample_data/S00775'),
  'new': PosixPath('/mnt/newStor/paros/paros_WORK/xintong/test/new_pipeline/S00775'),
  'comparison': PosixPath('/mnt/newStor/paros/paros_WORK/xintong/test/comparison/S00775'),
  'results': PosixPath('/mnt/newStor/paros/paros_WORK/xintong/code/cortical_trajectory_analysis/validation/results/S00775')}}

In [26]:
summary_columns = [
    "subject",
    "stage",
    "status",
    "metric",
    "value",
    "notes",
]

validation_summary = pd.DataFrame(
    columns=summary_columns
)

validation_summary

,subject,stage,status,metric,value,notes


In [27]:
summary_path = RESULTS_ROOT / "validation_summary.csv"

validation_summary.to_csv(
    summary_path,
    index=False,
)

print("Saved:", summary_path)

Saved: /mnt/newStor/paros/paros_WORK/xintong/code/cortical_trajectory_analysis/validation/results/validation_summary.csv


# 1. Pair Validation

In [28]:
from scipy.io import loadmat
import numpy as np
import pandas as pd


def load_pair_mat(mat_path):
    """
    Load pair coordinates from a MATLAB .mat file.

    Expected format:
        N x 6
        [white_x, white_y, white_z, pial_x, pial_y, pial_z]
    """
    mat_path = Path(mat_path)

    if not mat_path.is_file():
        raise FileNotFoundError(f"Pair file not found: {mat_path}")

    data = loadmat(mat_path)

    candidates = []

    for key, value in data.items():
        if key.startswith("__"):
            continue

        if isinstance(value, np.ndarray):
            if value.ndim == 2 and value.shape[1] == 6:
                candidates.append((key, value))

    if len(candidates) == 0:
        raise ValueError(
            f"No N x 6 pair array found in {mat_path}"
        )

    if len(candidates) > 1:
        print(
            f"Warning: multiple N x 6 arrays found in {mat_path}: "
            f"{[key for key, _ in candidates]}"
        )

    key, pair = candidates[0]

    pair = np.asarray(pair, dtype=float)

    print(
        f"Loaded {mat_path.name}: "
        f"variable='{key}', shape={pair.shape}"
    )

    return pair

## a.Compare gemotry distance

In [29]:
def get_pair_statistics(pair):
    """
    Calculate basic pair statistics.
    """
    white = pair[:, :3]
    pial = pair[:, 3:6]

    distances = np.linalg.norm(
        pial - white,
        axis=1,
    )

    stats = {
        "n_trajectories": pair.shape[0],

        "white_x_min": white[:, 0].min(),
        "white_x_max": white[:, 0].max(),
        "white_y_min": white[:, 1].min(),
        "white_y_max": white[:, 1].max(),
        "white_z_min": white[:, 2].min(),
        "white_z_max": white[:, 2].max(),

        "pial_x_min": pial[:, 0].min(),
        "pial_x_max": pial[:, 0].max(),
        "pial_y_min": pial[:, 1].min(),
        "pial_y_max": pial[:, 1].max(),
        "pial_z_min": pial[:, 2].min(),
        "pial_z_max": pial[:, 2].max(),

        "distance_mean": distances.mean(),
        "distance_median": np.median(distances),
        "distance_std": distances.std(),
        "distance_min": distances.min(),
        "distance_max": distances.max(),
    }

    return stats

## b. Compare the coordinates of pair

In [30]:
def compare_pairs(old_pair, new_pair):
    """
    Compare old and new pair coordinates.

    Point-wise comparison is only performed when
    both arrays have the same shape.
    """
    result = {
        "old_n": old_pair.shape[0],
        "new_n": new_pair.shape[0],
        "same_shape": old_pair.shape == new_pair.shape,
    }

    if old_pair.shape != new_pair.shape:
        result.update({
            "mean_coordinate_error": np.nan,
            "median_coordinate_error": np.nan,
            "p95_coordinate_error": np.nan,
            "max_coordinate_error": np.nan,
        })

        return result

    old_white = old_pair[:, :3]
    old_pial = old_pair[:, 3:6]

    new_white = new_pair[:, :3]
    new_pial = new_pair[:, 3:6]

    white_error = np.linalg.norm(
        old_white - new_white,
        axis=1,
    )

    pial_error = np.linalg.norm(
        old_pial - new_pial,
        axis=1,
    )

    all_errors = np.concatenate(
        [white_error, pial_error]
    )

    result.update({
        "white_mean_error": white_error.mean(),
        "white_max_error": white_error.max(),

        "pial_mean_error": pial_error.mean(),
        "pial_max_error": pial_error.max(),

        "mean_coordinate_error": all_errors.mean(),
        "median_coordinate_error": np.median(all_errors),
        "p95_coordinate_error": np.percentile(
            all_errors,
            95,
        ),
        "max_coordinate_error": all_errors.max(),
    })

    return result

In [31]:
def validate_pair(
    subject,
    hemisphere,
    old_pair_path,
    new_pair_path,
    results_root,
):
    """
    Validate one old/new pair file pair.
    """

    print()
    print(
        f"Pair validation: "
        f"{subject} {hemisphere.upper()}"
    )
    print()

    old_pair = load_pair_mat(old_pair_path)
    new_pair = load_pair_mat(new_pair_path)

    old_stats = get_pair_statistics(old_pair)
    new_stats = get_pair_statistics(new_pair)

    comparison = compare_pairs(
        old_pair,
        new_pair,
    )

    result = {
        "subject": subject,
        "hemisphere": hemisphere,

        "old_shape": str(old_pair.shape),
        "new_shape": str(new_pair.shape),

        "old_n": old_pair.shape[0],
        "new_n": new_pair.shape[0],

        "old_distance_mean": old_stats["distance_mean"],
        "new_distance_mean": new_stats["distance_mean"],

        "old_distance_median": old_stats["distance_median"],
        "new_distance_median": new_stats["distance_median"],

        **comparison,
    }

    result_df = pd.DataFrame([result])

    subject_result_dir = (
        results_root
        / subject
        / "pair"
    )

    subject_result_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    output_path = (
        subject_result_dir
        / f"{hemisphere}_pair_validation.csv"
    )

    result_df.to_csv(
        output_path,
        index=False,
    )

    print()
    print(result_df.T)
    print()
    print("Saved:", output_path)

    return result_df

## c. create path

In [32]:
def get_validation_paths(subject):
    """
    Build old and new file paths for validation.
    """

    old_1mm = (
        DATA_ROOT
        / subject
        / "columns_1mm"
    )

    old_columns = (
        DATA_ROOT
        / subject
        / "columns"
    )

    new_trajectories = (
        TEST_ROOT
        / "new_pipeline"
        / subject
        / "trajectories"
    )

    return {
        "pair": {
            "lh": {
                "old": old_1mm / f"{subject}_pair_lh.mat",
                "new": new_trajectories / f"{subject}_pair_lh.mat",
            },
            "rh": {
                "old": old_1mm / f"{subject}_pair_rh.mat",
                "new": new_trajectories / f"{subject}_pair_rh.mat",
            },
        },

        "trajectory": {
            "lh": {
                "old": old_columns / f"{subject}_column_lh.mat",
                "new": new_trajectories / f"{subject}_trajectory_lh.mat",
            },
            "rh": {
                "old": old_columns / f"{subject}_column_rh.mat",
                "new": new_trajectories / f"{subject}_trajectory_rh.mat",
            },
        },
    }

In [33]:
FILES = {
    subject: get_validation_paths(subject)
    for subject in SUBJECTS
}

In [37]:
def check_pair_files():
    """
    Check whether old and new pair files exist
    for all selected subjects.
    """

    for subject in SUBJECTS:

        print(f"\nSubject: {subject}")

        for hemisphere in ["lh", "rh"]:

            paths = FILES[subject]["pair"][hemisphere]

            print(f"\n  {hemisphere.upper()}")

            print(
                "  Old:",
                paths["old"].is_file(),
                paths["old"],
            )

            print(
                "  New:",
                paths["new"].is_file(),
                paths["new"],
            )

In [38]:
check_pair_files()


Subject: S00775

  LH
  Old: True /mnt/newStor/paros/paros_WORK/xintong/data/column_sample_data/S00775/columns_1mm/S00775_pair_lh.mat
  New: True /mnt/newStor/paros/paros_WORK/xintong/test/new_pipeline/S00775/trajectories/S00775_pair_lh.mat

  RH
  Old: True /mnt/newStor/paros/paros_WORK/xintong/data/column_sample_data/S00775/columns_1mm/S00775_pair_rh.mat
  New: True /mnt/newStor/paros/paros_WORK/xintong/test/new_pipeline/S00775/trajectories/S00775_pair_rh.mat


In [39]:
pair_results = []

for subject in SUBJECTS:

    for hemisphere in ["lh", "rh"]:

        paths = FILES[subject]["pair"][hemisphere]

        if not paths["old"].is_file():
            print(
                f"Skip {subject} {hemisphere}: "
                "old pair file not found"
            )
            continue

        if not paths["new"].is_file():
            print(
                f"Skip {subject} {hemisphere}: "
                "new pair file not found"
            )
            continue

        result = validate_pair(
            subject=subject,
            hemisphere=hemisphere,
            old_pair_path=paths["old"],
            new_pair_path=paths["new"],
            results_root=RESULTS_ROOT,
        )

        pair_results.append(result)


Pair validation: S00775 LH

Loaded S00775_pair_lh.mat: variable='lh_pair', shape=(121365, 6)
Loaded S00775_pair_lh.mat: variable='lh_pair', shape=(121365, 6)

                                   0
subject                       S00775
hemisphere                        lh
old_shape                (121365, 6)
new_shape                (121365, 6)
old_n                         121365
new_n                         121365
old_distance_mean            2.56847
new_distance_mean            2.56847
old_distance_median         2.640167
new_distance_median         2.640167
same_shape                      True
white_mean_error                 0.0
white_max_error                  0.0
pial_mean_error                  0.0
pial_max_error                   0.0
mean_coordinate_error            0.0
median_coordinate_error          0.0
p95_coordinate_error             0.0
max_coordinate_error             0.0

Saved: /mnt/newStor/paros/paros_WORK/xintong/code/cortical_trajectory_analysis/validation/results/S

In [40]:
if pair_results:

    pair_results = pd.concat(
        pair_results,
        ignore_index=True,
    )

    display(pair_results)

else:
    print("No pair validation results.")

,subject,hemisphere,old_shape,new_shape,old_n,new_n,old_distance_mean,new_distance_mean,old_distance_median,new_distance_median,same_shape,white_mean_error,white_max_error,pial_mean_error,pial_max_error,mean_coordinate_error,median_coordinate_error,p95_coordinate_error,max_coordinate_error
0,S00775,lh,"(121365, 6)","(121365, 6)",121365,121365,2.568470,2.568470,2.640167,2.640167,True,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,S00775,rh,"(123599, 6)","(123599, 6)",123599,123599,2.529764,2.529764,2.595788,2.595788,True,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# 2.Trajectory Validation

In [41]:
def load_trajectory_mat(mat_path):
    """
    Load trajectory/column coordinates from a MATLAB .mat file
    and convert them to shape (N, 21, 3).
    """

    mat_path = Path(mat_path)

    if not mat_path.is_file():
        raise FileNotFoundError(
            f"Trajectory file not found: {mat_path}"
        )

    data = loadmat(mat_path)

    candidates = []

    for key, value in data.items():
        if key.startswith("__"):
            continue

        if not isinstance(value, np.ndarray):
            continue

        if value.ndim != 2:
            continue

        # Old format: (N, 63)
        if value.shape[1] == 63:
            trajectory = value.reshape(-1, 21, 3)
            candidates.append((key, trajectory))

        # New format: (N*21, 3)
        elif value.shape[1] == 3 and value.shape[0] % 21 == 0:
            trajectory = value.reshape(-1, 21, 3)
            candidates.append((key, trajectory))

    if len(candidates) == 0:
        raise ValueError(
            f"No valid trajectory array found in {mat_path}"
        )

    key, trajectory = candidates[0]

    trajectory = np.asarray(
        trajectory,
        dtype=float,
    )

    print(
        f"Loaded {mat_path.name}: "
        f"variable='{key}', "
        f"shape={trajectory.shape}"
    )

    return trajectory

In [42]:
def compare_trajectories(old_trajectory, new_trajectory):
    """
    Compare old and new trajectories point by point.
    """

    result = {
        "old_shape": old_trajectory.shape,
        "new_shape": new_trajectory.shape,
        "same_shape": old_trajectory.shape == new_trajectory.shape,
    }

    if old_trajectory.shape != new_trajectory.shape:
        result.update({
            "mean_point_error": np.nan,
            "median_point_error": np.nan,
            "p95_point_error": np.nan,
            "max_point_error": np.nan,
        })

        return result

    point_error = np.linalg.norm(
        old_trajectory - new_trajectory,
        axis=2,
    )

    result.update({
        "mean_point_error": point_error.mean(),
        "median_point_error": np.median(point_error),
        "p95_point_error": np.percentile(
            point_error,
            95,
        ),
        "max_point_error": point_error.max(),
    })

    return result

In [43]:
def validate_trajectory(
    subject,
    hemisphere,
    old_path,
    new_path,
    results_root,
):
    """
    Validate one old/new trajectory file pair.
    """

    print()
    print(
        f"Trajectory validation: "
        f"{subject} {hemisphere.upper()}"
    )
    print()

    old_trajectory = load_trajectory_mat(old_path)
    new_trajectory = load_trajectory_mat(new_path)

    comparison = compare_trajectories(
        old_trajectory,
        new_trajectory,
    )

    result = {
        "subject": subject,
        "hemisphere": hemisphere,
        "old_n": old_trajectory.shape[0],
        "new_n": new_trajectory.shape[0],
        "points_per_trajectory": old_trajectory.shape[1],
        **comparison,
    }

    result_df = pd.DataFrame([result])

    output_dir = (
        results_root
        / subject
        / "trajectory"
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    output_path = (
        output_dir
        / f"{hemisphere}_trajectory_validation.csv"
    )

    result_df.to_csv(
        output_path,
        index=False,
    )

    print(result_df.T)
    print()
    print("Saved:", output_path)

    return result_df

In [44]:
trajectory_results = []

for subject in SUBJECTS:

    for hemisphere in ["lh", "rh"]:

        paths = FILES[subject]["trajectory"][hemisphere]

        if not paths["old"].is_file():
            print(
                f"Skip {subject} {hemisphere}: "
                "old trajectory file not found"
            )
            continue

        if not paths["new"].is_file():
            print(
                f"Skip {subject} {hemisphere}: "
                "new trajectory file not found"
            )
            continue

        result = validate_trajectory(
            subject=subject,
            hemisphere=hemisphere,
            old_path=paths["old"],
            new_path=paths["new"],
            results_root=RESULTS_ROOT,
        )

        trajectory_results.append(result)


Trajectory validation: S00775 LH

Loaded S00775_column_lh.mat: variable='lh_column_points', shape=(121365, 21, 3)
Loaded S00775_trajectory_lh.mat: variable='lh_trajectory_points', shape=(121365, 21, 3)
                                     0
subject                         S00775
hemisphere                          lh
old_n                           121365
new_n                           121365
points_per_trajectory               21
old_shape              (121365, 21, 3)
new_shape              (121365, 21, 3)
same_shape                        True
mean_point_error                   0.0
median_point_error                 0.0
p95_point_error                    0.0
max_point_error                    0.0

Saved: /mnt/newStor/paros/paros_WORK/xintong/code/cortical_trajectory_analysis/validation/results/S00775/trajectory/lh_trajectory_validation.csv

Trajectory validation: S00775 RH

Loaded S00775_column_rh.mat: variable='rh_column_points', shape=(123599, 21, 3)
Loaded S00775_trajectory_rh.m

In [45]:
if trajectory_results:

    trajectory_results = pd.concat(
        trajectory_results,
        ignore_index=True,
    )

    display(trajectory_results)

else:
    print("No trajectory validation results.")

,subject,hemisphere,old_n,new_n,points_per_trajectory,old_shape,new_shape,same_shape,mean_point_error,median_point_error,p95_point_error,max_point_error
0,S00775,lh,121365,121365,21,"(121365, 21, 3)","(121365, 21, 3)",True,1.867776e-15,0.0,7.324107e-15,1.740467e-14
1,S00775,rh,123599,123599,21,"(123599, 21, 3)","(123599, 21, 3)",True,1.881979e-15,0.0,7.324107e-15,1.740467e-14


# 3.Registration Validation

## a.Transform to DWI voxel space

In [47]:
def validate_transformed_trajectories():
    results = []

    for subject in SUBJECTS:
        for hemisphere in ["lh", "rh"]:

            old_path = (
                DATA_ROOT
                / subject
                / "columns_1mm"
                / f"{subject}_column_{hemisphere}_dwi.mat"
            )

            new_path = (
                TEST_ROOT
                / "new_pipeline"
                / subject
                / "trajectories"
                / f"{subject}_trajectory_{hemisphere}_dwi.mat"
            )

            if not old_path.is_file() or not new_path.is_file():
                print(f"Skip {subject} {hemisphere}")
                continue

            old_data = loadmat(old_path)
            new_data = loadmat(new_path)

            old_arr = next(
                v for k, v in old_data.items()
                if not k.startswith("__")
                and isinstance(v, np.ndarray)
            )

            new_arr = next(
                v for k, v in new_data.items()
                if not k.startswith("__")
                and isinstance(v, np.ndarray)
            )

            # Both should represent homogeneous DWI voxel coordinates
            if old_arr.shape[0] == 4:
                old_xyz = old_arr[:3].T
            else:
                old_xyz = old_arr.reshape(-1, 3)

            if new_arr.shape[0] == 4:
                new_xyz = new_arr[:3].T
            else:
                new_xyz = new_arr.reshape(-1, 3)

            same_shape = old_xyz.shape == new_xyz.shape

            if same_shape:
                error = np.linalg.norm(
                    old_xyz - new_xyz,
                    axis=1,
                )

                result = {
                    "subject": subject,
                    "hemisphere": hemisphere,
                    "same_shape": True,
                    "mean_error": error.mean(),
                    "p95_error": np.percentile(error, 95),
                    "max_error": error.max(),
                }

            else:
                result = {
                    "subject": subject,
                    "hemisphere": hemisphere,
                    "same_shape": False,
                    "mean_error": np.nan,
                    "p95_error": np.nan,
                    "max_error": np.nan,
                }

            results.append(result)

    return pd.DataFrame(results)


transform_results = validate_transformed_trajectories()
display(transform_results)

,subject,hemisphere,same_shape,mean_error,p95_error,max_error
0,S00775,lh,True,0.000014,0.00003,0.000041
1,S00775,rh,True,0.000014,0.00003,0.000041


## b. Region Assignment Validation

Compared the new region-assigned trajectories with the original `label_coord_1mm` outputs for all 68 Desikan–Killiany cortical regions.
- Trajectory number & region


In [48]:
def validate_regions(subject):
    old_dir = DATA_ROOT / subject / "label_coord_1mm"
    new_dir = TEST_ROOT / "new_pipeline" / subject / "regions"

    rows = []

    for hemisphere in ["lh", "rh"]:
        for old_path in sorted(old_dir.glob(f"{hemisphere}_*.mat")):

            region = old_path.stem

            # Skip non-standard extra labels
            if "exvivo" in region or "thresh" in region:
                continue

            new_path = new_dir / f"{region}.mat"

            if not new_path.exists():
                rows.append({
                    "subject": subject,
                    "region": region,
                    "status": "MISSING_NEW",
                })
                continue

            old_data = loadmat(old_path)
            new_data = loadmat(new_path)

            old_arr = next(
                v for k, v in old_data.items()
                if not k.startswith("__")
                and isinstance(v, np.ndarray)
            )

            new_arr = next(
                v for k, v in new_data.items()
                if not k.startswith("__")
                and isinstance(v, np.ndarray)
            )

            same_shape = old_arr.shape == new_arr.shape

            if same_shape:
                error = np.linalg.norm(
                    old_arr[:3, :] - new_arr[:3, :],
                    axis=0,
                )

                mean_error = error.mean()
                max_error = error.max()

                n_old = old_arr.shape[1] // 21
                n_new = new_arr.shape[1] // 21

                passed = max_error < 1e-4

            else:
                mean_error = np.nan
                max_error = np.nan
                n_old = old_arr.shape[1] // 21
                n_new = new_arr.shape[1] // 21
                passed = False

            rows.append({
                "subject": subject,
                "region": region,
                "old_n": n_old,
                "new_n": n_new,
                "same_shape": same_shape,
                "mean_error": mean_error,
                "max_error": max_error,
                "status": "PASS" if passed else "FAIL",
            })

    return pd.DataFrame(rows)


region_results = validate_regions("S00775")

display(region_results)

print("\nSummary:")
print(region_results["status"].value_counts())

print("\nMaximum error:")
print(region_results["max_error"].max())

,subject,region,status,old_n,new_n,same_shape,mean_error,max_error
0,S00775,lh_FG1_mpm_vpnl,MISSING_NEW,NaN,NaN,NaN,NaN,NaN
1,S00775,lh_FG2_mpm_vpnl,MISSING_NEW,NaN,NaN,NaN,NaN,NaN
2,S00775,lh_FG3_mpm_vpnl,MISSING_NEW,NaN,NaN,NaN,NaN,NaN
3,S00775,lh_FG4_mpm_vpnl,MISSING_NEW,NaN,NaN,NaN,NaN,NaN
4,S00775,lh_bankssts,PASS,1541.0,1541.0,True,0.000008,0.000013
...,...,...,...,...,...,...,...,...
85,S00775,rh_superiorparietal,PASS,7025.0,7025.0,True,0.000017,0.000038
86,S00775,rh_superiortemporal,PASS,5752.0,5752.0,True,0.000012,0.000034
87,S00775,rh_supramarginal,PASS,5386.0,5386.0,True,0.000009,0.000022
88,S00775,rh_temporalpole,PASS,537.0,537.0,True,0.000027,0.000035



Summary:
status
PASS           68
MISSING_NEW    22
Name: count, dtype: int64

Maximum error:
4.139771610311982e-05


## c.Profile Sampling Validation
Compared trajectory-level 21-point MD profiles 

In [49]:
def validate_profile_sampling(subject):
    old_dir = DATA_ROOT / subject / "columns_1mm"
    new_dir = TEST_ROOT / "new_pipeline" / subject / "profiles"

    rows = []

    for hemisphere in ["lh", "rh"]:
        pattern = f"{subject}_{hemisphere}_*_cols_md.csv"

        for old_path in sorted(old_dir.glob(pattern)):
            region = old_path.name.replace(
                f"{subject}_{hemisphere}_", ""
            ).replace("_cols_md.csv", "")

            new_path = (
                new_dir
                / f"{subject}_{hemisphere}_{region}_MD.csv"
            )

            if not new_path.exists():
                rows.append({
                    "subject": subject,
                    "hemisphere": hemisphere,
                    "region": region,
                    "status": "MISSING_NEW",
                })
                continue

            old = pd.read_csv(old_path, header=None).values
            new = pd.read_csv(new_path, header=None).values

            same_shape = old.shape == new.shape

            if same_shape:
                error = np.abs(old - new)

                result = {
                    "subject": subject,
                    "hemisphere": hemisphere,
                    "region": region,
                    "old_shape": old.shape,
                    "new_shape": new.shape,
                    "same_shape": True,
                    "mean_error": error.mean(),
                    "p95_error": np.percentile(error, 95),
                    "max_error": error.max(),
                    "status": (
                        "PASS"
                        if error.max() < 1e-4
                        else "FAIL"
                    ),
                }

            else:
                result = {
                    "subject": subject,
                    "hemisphere": hemisphere,
                    "region": region,
                    "old_shape": old.shape,
                    "new_shape": new.shape,
                    "same_shape": False,
                    "mean_error": np.nan,
                    "p95_error": np.nan,
                    "max_error": np.nan,
                    "status": "FAIL",
                }

            rows.append(result)

    return pd.DataFrame(rows)


profile_results = validate_profile_sampling("S00775")

display(profile_results)

print("\nSummary:")
print(profile_results["status"].value_counts())

print("\nMaximum error:")
print(profile_results["max_error"].max())

,subject,hemisphere,region,old_shape,new_shape,same_shape,mean_error,p95_error,max_error,status
0,S00775,lh,bankssts,"(1541, 21)","(1541, 21)",True,0.000002,0.000006,0.000016,PASS
1,S00775,lh,caudalanteriorcingulate,"(876, 21)","(876, 21)",True,0.000002,0.000006,0.000025,PASS
2,S00775,lh,caudalmiddlefrontal,"(2770, 21)","(2770, 21)",True,0.000003,0.000011,0.000041,PASS
3,S00775,lh,cuneus,"(2103, 21)","(2103, 21)",True,0.000001,0.000005,0.000024,PASS
4,S00775,lh,entorhinal,"(585, 21)","(585, 21)",True,0.000006,0.000018,0.000067,PASS
...,...,...,...,...,...,...,...,...,...,...
63,S00775,rh,superiorparietal,"(7025, 21)","(7025, 21)",True,0.000003,0.000012,0.000041,PASS
64,S00775,rh,superiortemporal,"(5752, 21)","(5752, 21)",True,0.000002,0.000010,0.000055,PASS
65,S00775,rh,supramarginal,"(5386, 21)","(5386, 21)",True,0.000002,0.000007,0.000031,PASS
66,S00775,rh,temporalpole,"(537, 21)","(537, 21)",True,0.000006,0.000020,0.000047,PASS



Summary:
status
PASS    68
Name: count, dtype: int64

Maximum error:
8.327587861106656e-05


# 4. Clean bad data

In [50]:
def validate_cleaning(subject):
    old_dir = DATA_ROOT / subject / "columns_1mm"
    new_dir = TEST_ROOT / "new_pipeline" / subject / "profiles"

    rows = []

    for hemisphere in ["lh", "rh"]:
        pattern = f"{subject}_{hemisphere}_*_cols_md.csv"

        for old_path in sorted(old_dir.glob(pattern)):
            region = old_path.name.replace(
                f"{subject}_{hemisphere}_", ""
            ).replace("_cols_md.csv", "")

            new_path = (
                new_dir
                / f"{subject}_{hemisphere}_{region}_MD.csv"
            )

            if not new_path.exists():
                continue

            old = pd.read_csv(old_path, header=None).values
            new = pd.read_csv(new_path, header=None).values

            same_shape = old.shape == new.shape

            if same_shape:
                error = np.abs(old - new)

                mean_error = error.mean()
                max_error = error.max()
                status = "PASS" if max_error < 1e-4 else "FAIL"
            else:
                mean_error = np.nan
                max_error = np.nan
                status = "FAIL"

            rows.append({
                "subject": subject,
                "hemisphere": hemisphere,
                "region": region,
                "old_n": old.shape[0],
                "new_n": new.shape[0],
                "same_shape": same_shape,
                "mean_error": mean_error,
                "max_error": max_error,
                "status": status,
            })

    return pd.DataFrame(rows)


cleaning_results = validate_cleaning("S00775")

display(cleaning_results)

print("\nSummary:")
print(cleaning_results["status"].value_counts())

print("\nRegions with different trajectory counts:")
display(
    cleaning_results[
        cleaning_results["old_n"] != cleaning_results["new_n"]
    ]
)

,subject,hemisphere,region,old_n,new_n,same_shape,mean_error,max_error,status
0,S00775,lh,bankssts,1541,1541,True,0.000002,0.000016,PASS
1,S00775,lh,caudalanteriorcingulate,876,876,True,0.000002,0.000025,PASS
2,S00775,lh,caudalmiddlefrontal,2770,2770,True,0.000003,0.000041,PASS
3,S00775,lh,cuneus,2103,2103,True,0.000001,0.000024,PASS
4,S00775,lh,entorhinal,585,585,True,0.000006,0.000067,PASS
...,...,...,...,...,...,...,...,...,...
63,S00775,rh,superiorparietal,7025,7025,True,0.000003,0.000041,PASS
64,S00775,rh,superiortemporal,5752,5725,False,NaN,NaN,FAIL
65,S00775,rh,supramarginal,5386,5386,True,0.000002,0.000031,PASS
66,S00775,rh,temporalpole,537,521,False,NaN,NaN,FAIL



Summary:
status
PASS    57
FAIL    11
Name: count, dtype: int64

Regions with different trajectory counts:


,subject,hemisphere,region,old_n,new_n,same_shape,mean_error,max_error,status
6,S00775,lh,fusiform,3981,3980,False,NaN,NaN,FAIL
8,S00775,lh,inferiortemporal,3922,3911,False,NaN,NaN,FAIL
15,S00775,lh,middletemporal,4038,4037,False,NaN,NaN,FAIL
30,S00775,lh,superiortemporal,5667,5665,False,NaN,NaN,FAIL
32,S00775,lh,temporalpole,496,494,False,NaN,NaN,FAIL
38,S00775,rh,entorhinal,491,490,False,NaN,NaN,FAIL
40,S00775,rh,fusiform,3928,3903,False,NaN,NaN,FAIL
42,S00775,rh,inferiortemporal,3656,3647,False,NaN,NaN,FAIL
49,S00775,rh,middletemporal,4527,4523,False,NaN,NaN,FAIL
64,S00775,rh,superiortemporal,5752,5725,False,NaN,NaN,FAIL


In [53]:
def validate_regional_profiles(subject):
    old_dir = DATA_ROOT / subject / "columns_1mm"
    new_dir = (
        TEST_ROOT
        / "new_pipeline"
        / subject
        / "regional_profiles"
        / "region_mean"
    )

    rows = []

    for new_path in sorted(new_dir.glob(f"{subject}_*_MD_mean.csv")):

        name = new_path.name

        parts = name.replace(
            f"{subject}_", ""
        ).replace(
            "_MD_mean.csv", ""
        ).split("_", 1)

        hemisphere = parts[0]
        region = parts[1]

        old_path = (
            old_dir
            / f"{subject}_{hemisphere}_{region}_md.csv"
        )

        if not old_path.exists():
            rows.append({
                "subject": subject,
                "hemisphere": hemisphere,
                "region": region,
                "status": "MISSING_OLD",
            })
            continue

        old = pd.read_csv(
            old_path,
            header=None,
        ).values.reshape(-1)

        new = pd.read_csv(
            new_path,
            header=None,
        ).values.reshape(-1)

        old_profile = old[:21]
        new_profile = new[:21]

        error = np.abs(
            old_profile - new_profile
        )

        rows.append({
            "subject": subject,
            "hemisphere": hemisphere,
            "region": region,
            "same_shape": old_profile.shape == new_profile.shape,
            "mean_error": error.mean(),
            "max_error": error.max(),
            "status": (
                "PASS"
                if error.max() < 1e-4
                else "FAIL"
            ),
        })

    return pd.DataFrame(rows)


regional_results = validate_regional_profiles("S00775")

display(regional_results)

print("\nSummary:")
print(regional_results["status"].value_counts())

print("\nFailed regions:")
display(
    regional_results[
        regional_results["status"] != "PASS"
    ]
)

,subject,hemisphere,region,same_shape,mean_error,max_error,status
0,S00775,lh,bankssts,True,1.946265e-07,5.043040e-07,PASS
1,S00775,lh,caudalanteriorcingulate,True,5.725132e-07,1.062787e-06,PASS
2,S00775,lh,caudalmiddlefrontal,True,8.220606e-07,4.778654e-06,PASS
3,S00775,lh,cuneus,True,3.887180e-08,7.501068e-08,PASS
4,S00775,lh,entorhinal,True,5.206758e-07,1.372949e-06,PASS
...,...,...,...,...,...,...,...
63,S00775,rh,superiorparietal,True,1.088671e-06,2.473043e-06,PASS
64,S00775,rh,superiortemporal,True,2.523426e-03,6.143331e-03,FAIL
65,S00775,rh,supramarginal,True,5.843812e-07,1.870418e-06,PASS
66,S00775,rh,temporalpole,True,1.790834e-02,3.118284e-02,FAIL



Summary:
status
PASS    57
FAIL    11
Name: count, dtype: int64

Failed regions:


,subject,hemisphere,region,same_shape,mean_error,max_error,status
6,S00775,lh,fusiform,True,0.000095,0.000265,FAIL
8,S00775,lh,inferiortemporal,True,0.000917,0.002245,FAIL
15,S00775,lh,middletemporal,True,0.000072,0.000253,FAIL
30,S00775,lh,superiortemporal,True,0.000159,0.000483,FAIL
32,S00775,lh,temporalpole,True,0.001715,0.004705,FAIL
38,S00775,rh,entorhinal,True,0.000740,0.001539,FAIL
40,S00775,rh,fusiform,True,0.004375,0.005340,FAIL
42,S00775,rh,inferiortemporal,True,0.001539,0.001774,FAIL
49,S00775,rh,middletemporal,True,0.000353,0.000953,FAIL
64,S00775,rh,superiortemporal,True,0.002523,0.006143,FAIL


In [56]:
old = pd.read_csv(
    TEST_ROOT
    / "existing_pipeline/output/S00775/adc/adc_cols_by_column/"
      "S00775_lh_bankssts_cols_adc.csv",
    header=None,
).values

new = pd.read_csv(
    TEST_ROOT
    / "new_pipeline/S00775/profiles/"
      "S00775_lh_bankssts_MD.csv",
    header=None,
).values

print("OLD ADC:")
print(old[0, :5])
print("mean =", old.mean())

print("\nNEW MD:")
print(new[0, :5])
print("mean =", new.mean())

OLD ADC:
[0.00067093 0.00061464 0.00053742 0.00046737 0.00040456]
mean = 0.0008144087836298966

NEW MD:
[0.53463164 0.56879492 0.60168404 0.63324066 0.66340647]
mean = 0.8278361843072828
